<div style="background:#C1ABA6">
<div style="font-size: xx-large ; font-weight: 900 ;  padding-top: 100px ; color: rgba(0 , 0 , 0 , 0.8); line-height: 100%"; align="center">FDSN web servisi</div>
    
---
<div align="center"> Vježbe iz Seizmologije I </div>
<div align="center"> ak. god. 2026./2027. </div>
<div align="center"> dr.sc. Katarina Zailac </div>

---
</div>

# FDSN web-servisi

U biblioteci [ObsPy](https://docs.obspy.org/) su definirani klijenti za dohvat podataka iz različitih izvora, između ostalih i klijente koji mogu dohvaćati podatke korištenjem [FDSN web servisa](https://www.fdsn.org/webservices/) (IRIS, Orfeus, USGS, ...).

Trenutno FDSN web servisi omogućuju dohvat podataka o:

* seizmičkim stanicama (u FDSN Station XML formatu)
* potresima (u QuakeML formatu)
* zapisima (MSEED format)

## Dohvat zapisa potresa

Dohvatit ćemo zapis Tohoku potresa s IRIS online baze. Koristit ćemo [FDSN Client](https://docs.obspy.org/packages/autogen/obspy.clients.fdsn.client.Client.html#obspy.clients.fdsn.client.Client). Zapisi potresa se dohvaćaju korištenjem metode [`get_waveforms()`](https://docs.obspy.org/packages/autogen/obspy.clients.fdsn.client.Client.get_waveforms.html#obspy.clients.fdsn.client.Client.get_waveforms).

In [ ]:
from obspy import UTCDateTime
from obspy.clients.fdsn import Client

client = Client("IRIS")
t = UTCDateTime("2011-03-11T05:46:23")  # Tohoku
st = client.get_waveforms("II", "PFO", "*", "LHZ", t + 10 * 60, t + 30 * 60)
print(st)
st.plot();

Zapisi su spremljeni u objekt tipa [`Stream`](https://docs.obspy.org/packages/autogen/obspy.core.stream.Stream.html#obspy.core.stream.Stream), jednako kako se sprema u objekt istog tipa ako se učitava iz lokalne datoteke.

Sve metode koje smo koristili na prvim vježbama vrijede i ovdje.

## Dohvat podataka o potresima (katalog)

Istu [`Client`](https://docs.obspy.org/packages/autogen/obspy.clients.fdsn.client.Client.html#obspy.clients.fdsn.client.Client) klasu možemo koristiti i za dohvat kataloga. 

Budući da `IRIS` baza više ne podržava eventove, morat ćemo inicijalizirati Client tako da koristi `GEOFON` bazu. Katalog se dohvaća korištenjem metode [`get_events()`](https://docs.obspy.org/packages/autogen/obspy.clients.fdsn.client.Client.get_events.html#obspy.clients.fdsn.client.Client.get_events).

In [ ]:
evt_client = Client("GEOFON")

catalog = evt_client.get_events(starttime=t - 100, endtime=t + 24 * 3600, minmagnitude=7)
print(catalog)
catalog.plot();

## Dohvat podataka o stanicama

Također se mogu dohvatiti i meta-podaci o stanicama (Station XML). Za to se koristi metoda [`get_stations()`](https://docs.obspy.org/packages/autogen/obspy.clients.fdsn.client.Client.get_stations.html#obspy.clients.fdsn.client.Client.get_stations). 

In [ ]:
event = catalog[0]
origin = event.origins[0]

# Münster
lon = 7.63
lat = 51.96

inventory = client.get_stations(longitude=lon, latitude=lat, maxradius=2.5, level="station")
print(inventory)

Parametar `level` služi za definiranje koliko detaljan Station XML želimo dohvatiti. Opcije su sljedeće:

* `"network"` - dohvaćaju se samo podaci o mrežama koje zadovoljavaju kriterije pretrage
* `"station"` - dohvaćaju se i svi podaci o stanicama koje zadovoljavaju kriterije pretrage
* `"channel"` - dohvaćaju se i podaci o svim kanalima svih stanica u svim mrežama koje zadovoljavaju kriterije pretrage
* `"response"` - dohvaćaju se i podaci o odzivima pojedinih kanala (velika količina podataka!)

In [ ]:
eida = Client("ODC")
inventory = eida.get_stations(network="CR", station="ZAG", level="station")
print(inventory)

In [ ]:
eida = Client("ODC")
inventory = eida.get_stations(network="CR", station="ZAG", level="channel")
print(inventory)

Za svaku od pokazanih metoda podaci se mogu direktno spremiti u datoteku zadavanjem parametra `filename`. U tom slučaju će podaci biti spremljeni u zadanu datoteku i mogu se koristiti i kasnije.

# Zadaci

## Zadatak 1

Dohvatite katalog potresa za 2016. godinu (možete koristiti GEOFON bazu) za nama zanimljivo područje (ugrubo između 10° i 20° geografske dužine te između 40° i 48° geografske širine). Pronađite najveći potres u katalogu.

## Zadatak 2

Dohvatite meta-podatke o postajama u privremenoj mreži koda `Z3`. Nacrtajte ih. Koristite `"ETH"` kao klijent.

## Zadatak 3

Dohvatite zapis najvećeg potresa dobivenog u [Zadatku 1](#zadatak-1) za postaju koda `A252A` te ga nacrtajte. Možete koristiti isti klijent kao u [Zadatku 2](#zadatak-2).